<a href="https://colab.research.google.com/github/atshaya1208/imdb_movies_prj/blob/main/Project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/content/imdb_movies_2024.csv')


In [2]:
df

,title,storyline
0,1. The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...
1,2. Gladiator II,After his home is conquered by the tyrannical ...
2,3. The Substance,A fading celebrity takes a black-market drug: ...
3,4. Milk & Serial,A surprise birthday prank takes a turn for the...
4,5. Dune: Part Two,Paul Atreides unites with the Fremen while on ...
...,...,...
5545,5546. Sree Muthappan,The film unfolds the mythology of Sree Muthapp...
5546,5547. Gorre Puranam,A sheep named Ram dreams of reaching a perfect...
5547,5548. Riki Rhino: The Bird Kingdom,"A fun, family adventure that follows a brave r..."
5548,5549. The Prey,A cult known as the Wendigo Clan hunts down va...


In [3]:
df.isnull().sum()

,0
title,0
storyline,0


In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.dtypes

,0
title,object
storyline,object


In [6]:
df.values[0]

array(['1. The Ministry of Ungentlemanly Warfare',
       'The British military recruits a small group of highly skilled soldiers to strike against German forces behind enemy lines during World War II.'],
      dtype=object)

In [7]:
import re

df['title'] = df['title'].str.replace(r'^\d+\.\s*', '', regex=True)

In [8]:
df['title']

,title
0,The Ministry of Ungentlemanly Warfare
1,Gladiator II
2,The Substance
3,Milk & Serial
4,Dune: Part Two
...,...
5545,Sree Muthappan
5546,Gorre Puranam
5547,Riki Rhino: The Bird Kingdom
5548,The Prey


In [9]:
df = df[df['storyline'] != 'No storyline available.'].copy()

In [10]:
df['storyline_processed'] = df['storyline'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

In [11]:
df['storyline_processed'] = df['storyline_processed'].str.lower()

In [12]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk import word_tokenize

stop_words = stopwords.words('english')

add_words = ['movie','br','go','film','ugh','one','make','even','see','movies','get','makes','making','time','watch','character','s']

stop_words.extend(add_words)

def remove_stopwords(story):
    storyline_tokenized = word_tokenize(story)
    storyline_new = " ".join([word for word in storyline_tokenized  if word not in stop_words])
    return storyline_new

df['storyline_processed'] = df['storyline_processed'].apply(remove_stopwords)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
df['storyline_processed']

,storyline_processed
0,british military recruits small group highly s...
1,home conquered tyrannical emperors lead rome l...
2,fading celebrity takes blackmarket drug cellre...
3,surprise birthday prank takes turn worse popul...
4,paul atreides unites fremen warpath revenge co...
...,...
5544,little boy encounters ghost deceased woman ple...
5545,unfolds mythology sree muthappan folklore god ...
5546,sheep named ram dreams reaching perfect spot l...
5547,fun family adventure follows brave rhino named...


In [14]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def nltk_tag_to_wordnet_tag(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None


def lemmatize_sentence(sentence):
    nltk_tagged = nltk.pos_tag(nltk.word_tokenize(sentence))
    wordnet_tagged = map(lambda x: (x[0], nltk_tag_to_wordnet_tag(x[1])), nltk_tagged)

    lemmatized_sentence = []

    for word, tag in wordnet_tagged:
        if tag is None:
            lemmatized_sentence.append(word)
        else:
            lemmatized_sentence.append(lemmatizer.lemmatize(word, tag))
    return " ".join(lemmatized_sentence)


df['storyline_processed'] = df['storyline_processed'].apply(lambda x: lemmatize_sentence(x))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [15]:
df

,title,storyline,storyline_processed
0,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,british military recruit small group highly sk...
1,Gladiator II,After his home is conquered by the tyrannical ...,home conquer tyrannical emperor lead rome luci...
2,The Substance,A fading celebrity takes a black-market drug: ...,fade celebrity take blackmarket drug cellrepli...
3,Milk & Serial,A surprise birthday prank takes a turn for the...,surprise birthday prank take turn worse popula...
4,Dune: Part Two,Paul Atreides unites with the Fremen while on ...,paul atreides unites fremen warpath revenge co...
...,...,...,...
5544,Bhootpori,A little boy encounters the ghost of a decease...,little boy encounter ghost decease woman plead...
5545,Sree Muthappan,The film unfolds the mythology of Sree Muthapp...,unfolds mythology sree muthappan folklore god ...
5546,Gorre Puranam,A sheep named Ram dreams of reaching a perfect...,sheep name ram dream reach perfect spot long a...
5547,Riki Rhino: The Bird Kingdom,"A fun, family adventure that follows a brave r...",fun family adventure follow brave rhino name r...


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

tfidf = TfidfVectorizer(max_features = 5000)

X = tfidf.fit_transform(df['storyline_processed']).toarray()

similarity_matrix = cosine_similarity(X, X)

In [17]:
import pickle

with open("movie_df.pkl", "wb") as f:
    pickle.dump(df, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(X, f)